In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score

# Models to evaluate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
dataset_path = '../data/personality_dataset.csv'
df = pd.read_csv(dataset_path)
print(f"Dataset loaded. Shape: {df.shape}")

Dataset loaded. Shape: (2900, 8)


In [3]:
target_col = 'Personality'
X = df.drop(columns=[target_col])
y = df[target_col]

# Map targets to binary (0: Introvert, 1: Extrovert)
y_mapped = y.map({'Introvert': 0, 'Extrovert': 1})

# Define feature types
num_features = ['Time_spent_Alone', 'Social_event_attendance', 'Going_outside', 'Friends_circle_size', 'Post_frequency']
cat_features = ['Stage_fear', 'Drained_after_socializing']

# Build the transformers
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

# Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_mapped, test_size=0.2, random_state=42, stratify=y_mapped
)

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
}

print("Running 5-Fold Cross-Validation...")
results = []

for name, model in models.items():
    fold_accuracies = []
    fold_f1s = []
    
    for train_idx, val_idx in cv.split(X_train, y_train):
        # Split folds
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Build and train pipeline for this fold
        fold_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])
        fold_pipeline.fit(X_tr, y_tr)
        
        # Predict on validation fold
        val_pred = fold_pipeline.predict(X_val)
        
        fold_accuracies.append(accuracy_score(y_val, val_pred))
        fold_f1s.append(f1_score(y_val, val_pred))
        
    avg_acc = np.mean(fold_accuracies)
    avg_f1 = np.mean(fold_f1s)
    
    results.append({
        'Model': name,
        'Mean CV Accuracy': avg_acc,
        'Mean CV F1-Score': avg_f1
    })

results_df = pd.DataFrame(results)
print("\n=== Model Comparison Results ===")
print(results_df.to_string(index=False))


Running 5-Fold Cross-Validation...


c:\Personality_prediction\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [09:01:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Personality_prediction\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [09:01:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Personality_prediction\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [09:01:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Personality_prediction\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [09:01:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not


=== Model Comparison Results ===
              Model  Mean CV Accuracy  Mean CV F1-Score
Logistic Regression          0.931034          0.932522
      Random Forest          0.924138          0.926029
            XGBoost          0.918966          0.920624


c:\Personality_prediction\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [09:01:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
print("\nTuning Hyperparameters for Logistic Regression (Our Champion)...")

# Define the full pipeline with Logistic Regression as the classifier
tuning_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Define parameter grid for Logistic Regression
# 'C' is regularization strength (smaller values mean stronger regularization)
# 'penalty' defines the type of regularization
param_grid = {
    'classifier__C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'classifier__penalty': ['l2'] # standard l2 penalty
}

# Run Grid Search
grid_search = GridSearchCV(tuning_pipeline, param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_tuned_pipeline = grid_search.best_estimator;
print(f"Optimal Hyperparameters: {grid_search.best_params_}")


Tuning Hyperparameters for Random Forest...
Optimal Hyperparameters: {'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 50}


In [ ]:
test_preds = best_tuned_pipeline.predict(X_test)
test_acc = accuracy_score(y_test, test_preds)
print(f"\nFinal Test Holdout Accuracy: {test_acc:.4f}")
print("\nClassification Report on Holdout Set:\n", classification_report(y_test, test_preds, target_names=['Introvert', 'Extrovert']))


Final Test Holdout Accuracy: 0.9241

Classification Report on Holdout Set:
               precision    recall  f1-score   support

   Introvert       0.91      0.94      0.92       282
   Extrovert       0.94      0.91      0.93       298

    accuracy                           0.92       580
   macro avg       0.92      0.92      0.92       580
weighted avg       0.92      0.92      0.92       580



In [ ]:
output_dir = '../model'
os.makedirs(output_dir, exist_ok=True)
save_path = os.path.join(output_dir, 'model_pipeline.pkl')

joblib.dump(best_tuned_pipeline, save_path)
print(f"\nChampion Logistic Regression pipeline successfully saved to: {save_path}")


Tuned model pipeline successfully saved to: ../model\model_pipeline.pkl
